<a href="https://colab.research.google.com/github/panhpham2000/Fresh-Retail/blob/Mandison/copy_of_freshretail_starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fresh Retail: Starter Notebook

This notebook accompanies the **Introduction** slide deck (`FreshRetail_Introduction.pptx`). It provides a shared data pipeline for both tracks, then produces the exact outputs previewed in the showcase slides.

**Run sections 1–4 first** (shared setup), then run the section for your track:

| Section | Track | What you produce |
|---------|-------|-----------------|
| 5. Operations | Ops | Temporal profiles, heatmaps, KPIs, hourly patterns |
| 6. Data Science | DS | WAPE baselines, forecast overlays, demand recovery, error analysis |

Both tracks use the same dataset, same helper functions, and same time split.

- **Operations Track**: O1 (Diagnosis) or O2 (Decision)
- **Data Science Track**: D1 (Direct benchmark) or D2 (Recovery first)

**Dataset**: [Dingdong-Inc/FreshRetailNet-50K](https://huggingface.co/datasets/Dingdong-Inc/FreshRetailNet-50K)

---
## 1. Setup and Data Download

In [ ]:
# Run this cell on Google Colab (already installed locally)
!pip install -q pandas pyarrow matplotlib seaborn datasets

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

print("Setup complete.")

In [ ]:
from datasets import load_dataset

print("Downloading FreshRetailNet-50K from Hugging Face...")
ds = load_dataset("Dingdong-Inc/FreshRetailNet-50K")
print(ds)

# Convert to pandas
train_raw = ds["train"].to_pandas()
eval_raw = ds["eval"].to_pandas()

print(f"\nTrain: {train_raw.shape}, Eval: {eval_raw.shape}")
print(f"Columns: {list(train_raw.columns)}")

---
## 2. Data Preparation

In [ ]:
def prepare_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Prepare the raw HF dataset into a clean analysis panel."""
    df = df.copy()

    # Parse date
    df["dt"] = pd.to_datetime(df["dt"])
    df = df.sort_values(["store_id", "product_id", "dt"]).reset_index(drop=True)

    # Create series_id (unique store x product combination)
    series_keys = df[["store_id", "product_id"]].drop_duplicates().reset_index(drop=True)
    series_keys["series_id"] = range(1, len(series_keys) + 1)
    df = df.merge(series_keys, on=["store_id", "product_id"], how="left")

    # Create day index (days since start)
    min_date = df["dt"].min()
    df["day_idx"] = (df["dt"] - min_date).dt.days + 1

    n_series = df["series_id"].nunique()
    n_days = df["day_idx"].nunique()
    print(f"Prepared {len(df):,} rows \u2014 {n_series:,} series x {n_days} days")
    print(f"Date range: {df['dt'].min().date()} to {df['dt'].max().date()}")
    return df


history = prepare_panel(train_raw)
history.head()

---
## 3. Shared Functions: flag_censoring, make_features, time_split

In [ ]:
def flag_censoring(df: pd.DataFrame) -> pd.DataFrame:
    """Add censoring flags based on stockout hours."""
    df = df.copy()
    df["is_censored"] = (df["stock_hour6_22_cnt"] > 0).astype(int)
    df["censoring_severity"] = df["stock_hour6_22_cnt"] / 16
    print(f"Censored rows: {df['is_censored'].sum():,} / {len(df):,} ({df['is_censored'].mean():.1%})")
    return df


def make_features(df: pd.DataFrame) -> pd.DataFrame:
    """Add lag and rolling features for EDA and forecasting."""
    df = df.sort_values(["series_id", "day_idx"]).copy()
    grp = df.groupby("series_id")["sale_amount"]
    df["sales_lag1"] = grp.shift(1)
    df["sales_lag7"] = grp.shift(7)
    df["sales_roll7"] = grp.transform(lambda x: x.rolling(7, min_periods=1).mean())
    df["sales_roll28"] = grp.transform(lambda x: x.rolling(28, min_periods=1).mean())
    df["psd"] = grp.transform("mean")  # per-series daily mean
    return df


def time_split(df: pd.DataFrame, horizon: int = 7) -> tuple:
    """Split into train and validation by time. Validation = last `horizon` days."""
    min_day = df["day_idx"].min()
    max_day = df["day_idx"].max()
    val_start = max_day - horizon + 1
    train = df[df["day_idx"] < val_start].copy()
    val = df[df["day_idx"] >= val_start].copy()
    print(f"Train: day {min_day}..{val_start - 1} ({len(train):,} rows), Val: day {val_start}..{max_day} ({len(val):,} rows)")
    return train, val

In [ ]:
# Apply shared pipeline
history = flag_censoring(history)
history = make_features(history)

train, val = time_split(history, horizon=7)
print(f"\nValidation window: day {val['day_idx'].min()} to {val['day_idx'].max()}")

---
## 4. Data at a Glance

In [ ]:
# Show a real series with stockouts
series_stockouts = history.groupby("series_id")["is_censored"].mean()
example_sid = series_stockouts[(series_stockouts > 0.3) & (series_stockouts < 0.7)].index[0]

s_example = history[history["series_id"] == example_sid][
    ["dt", "day_idx", "sale_amount", "stock_hour6_22_cnt", "is_censored", "discount", "holiday_flag", "avg_temperature"]
].head(14)
print(f"Series {example_sid} \u2014 first 14 days (a product with frequent stockouts):")
display(s_example)

In [ ]:
# Dataset dimensions
summary = pd.Series({
    "Total rows": f"{len(history):,}",
    "Series (store x product)": f"{history['series_id'].nunique():,}",
    "Days per series": str(history["day_idx"].nunique()),
    "Products (product_id)": str(history["product_id"].nunique()),
    "Stores (store_id)": str(history["store_id"].nunique()),
    "Cities (city_id)": str(history["city_id"].nunique()),
    "Management groups": str(history["management_group_id"].nunique()),
    "Mean daily sales": f"{history['sale_amount'].mean():.3f}",
    "Censored rows": f"{history['is_censored'].sum():,} ({history['is_censored'].mean():.1%})",
    "Low-sale series (psd<1)": f"{(history.groupby('series_id')['psd'].first() < 1).sum():,}",
    "High-sale series (psd>=1)": f"{(history.groupby('series_id')['psd'].first() >= 1).sum():,}",
})
display(summary.to_frame("Value"))

In [ ]:
# Sales distribution and per-series daily mean
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

clipped = history["sale_amount"].clip(upper=history["sale_amount"].quantile(0.99))
axes[0].hist(clipped, bins=50, color="#065A82", edgecolor="white")
axes[0].set_title("Distribution of daily sales (clipped at 99th pctl)")
axes[0].set_xlabel("sale_amount")
axes[0].set_ylabel("Count")

psd_vals = history.groupby("series_id")["psd"].first()
axes[1].hist(psd_vals, bins=50, color="#1C7293", edgecolor="white")
axes[1].axvline(1.0, color="#E74C3C", linestyle="--", linewidth=2, label="psd=1 cutoff")
axes[1].set_title("Per-series daily mean (psd) distribution")
axes[1].set_xlabel("psd")
axes[1].set_ylabel("Number of series")
axes[1].legend()

plt.tight_layout()
plt.show()

---
---
# OPERATIONS TRACK

## 5. Operations Track

> **You may skip this section if you are focusing on the Data Science track.**

This section produces the following outputs — each corresponds to a slide in the deck:

| Output | What it shows | Slide |
|--------|--------------|-------|
| Temporal profiles (3 line charts) | Mean sales, stockout hours, and censoring share over 90 days | Slide 10 |
| City × group heatmap | Where stockouts concentrate geographically | Slide 11 |
| Availability bar chart | Which cities have the worst service levels | Slide 11 |
| Hourly stockout bar chart | How stockouts accumulate through the day | Slide 12 |
| Dual-axis time series | One product's sales overlaid with stockout hours | Slide 12 |

**A strong operations project** extends these starter visualizations into a focused analysis that answers one clear question (O1 or O2).

### 5a. EDA Strategy: Where to Look

In [ ]:
# --- Temporal profiles ---
daily_profile = history.groupby("day_idx").agg(
    mean_sales=("sale_amount", "mean"),
    mean_stockout_hours=("stock_hour6_22_cnt", "mean"),
    censored_share=("is_censored", "mean"),
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(daily_profile["day_idx"], daily_profile["mean_sales"], color="#B85042", linewidth=1.5)
axes[0].set_title("Mean daily sales over time")
axes[0].set_xlabel("Day index")
axes[0].set_ylabel("Mean sale_amount")

axes[1].plot(daily_profile["day_idx"], daily_profile["mean_stockout_hours"], color="#E8913A", linewidth=1.5)
axes[1].set_title("Mean stockout hours over time")
axes[1].set_xlabel("Day index")
axes[1].set_ylabel("Mean stock_hour6_22_cnt")

axes[2].plot(daily_profile["day_idx"], daily_profile["censored_share"], color="#5B8C5A", linewidth=1.5)
axes[2].set_title("Share of censored rows over time")
axes[2].set_xlabel("Day index")
axes[2].set_ylabel("Censored share")

plt.tight_layout()
plt.show()

In [ ]:
# --- Heatmap: stockout hours by city x management group ---
heatmap_data = history.groupby(["city_id", "management_group_id"])["stock_hour6_22_cnt"].mean().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(heatmap_data, cmap="YlOrRd", annot=True, fmt=".1f", ax=ax, linewidths=0.5)
ax.set_title("Mean stockout hours by city x management group")
ax.set_xlabel("Management group")
ax.set_ylabel("City")
plt.tight_layout()
plt.show()

### 5b. EDA Strategy: What to Measure

In [ ]:
# --- Operational KPIs per series ---
series_kpis = history.groupby("series_id").agg(
    psd=("sale_amount", "mean"),
    stockout_frequency=("is_censored", "mean"),
    mean_stockout_hours=("stock_hour6_22_cnt", "mean"),
    availability_rate=("censoring_severity", "mean"),
    city=("city_id", "first"),
    mgmt_group=("management_group_id", "first"),
).reset_index()

# Convert from mean censoring severity to availability rate
series_kpis["availability_rate"] = 1 - series_kpis["availability_rate"]

print("=== Series-level KPI summary ===")
display(series_kpis[["psd", "stockout_frequency", "mean_stockout_hours", "availability_rate"]].describe().round(3))

In [ ]:
# --- Distribution of stockout frequency and availability ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(series_kpis["stockout_frequency"], bins=40, color="#B85042", edgecolor="white")
axes[0].set_title("Distribution of stockout frequency across series")
axes[0].set_xlabel("Fraction of days with stockout")
axes[0].set_ylabel("Number of series")

axes[1].hist(series_kpis["availability_rate"], bins=40, color="#A7BEAE", edgecolor="white")
axes[1].set_title("Distribution of daily availability rate across series")
axes[1].set_xlabel("Availability rate (fraction of hours available)")
axes[1].set_ylabel("Number of series")

plt.tight_layout()
plt.show()

In [ ]:
# --- Availability by city ---
city_avail = series_kpis.groupby("city")["availability_rate"].mean().sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
city_avail.plot(kind="barh", color="#A7BEAE", edgecolor="white", ax=ax)
ax.set_title("Mean availability rate by city")
ax.set_xlabel("Availability rate")
ax.set_ylabel("City ID")
plt.tight_layout()
plt.show()

### 5c. Hourly Stockout Patterns

In [ ]:
# Expand hourly stock status to compute hourly stockout rates
hourly_stock = np.stack(history["hours_stock_status"].values)
hourly_labels = [f"h{h:02d}" for h in range(24)]
hourly_stockout_rate = pd.Series(hourly_stock.mean(axis=0), index=hourly_labels)

fig, ax = plt.subplots(figsize=(12, 4))
hourly_stockout_rate.plot(kind="bar", color="#C0392B", edgecolor="white", ax=ax)
ax.set_title("Hourly stockout rate across all series and days")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fraction of observations stocked out")
plt.tight_layout()
plt.show()

### 5d. Example: One Product Over Time

In [ ]:
# Pick a high-sale series with some stockouts
high_stockout = series_kpis[(series_kpis["psd"] > 3) & (series_kpis["stockout_frequency"] > 0.3)]
if len(high_stockout) > 0:
    example_sid2 = high_stockout.iloc[0]["series_id"]
else:
    example_sid2 = history.groupby("series_id")["psd"].first().idxmax()

example = history[history["series_id"] == example_sid2].copy()
n_days = len(example)

fig, ax1 = plt.subplots(figsize=(14, 4))
ax1.plot(example["dt"], example["sale_amount"], color="#B85042", linewidth=2, label="Observed sales")
ax1.set_xlabel("Date")
ax1.set_ylabel("sale_amount", color="#B85042")
ax1.tick_params(axis="y", labelcolor="#B85042")

ax2 = ax1.twinx()
ax2.fill_between(example["dt"], 0, example["stock_hour6_22_cnt"], alpha=0.3, color="#E8913A", label="Stockout hours")
ax2.set_ylabel("Stockout hours", color="#E8913A")
ax2.tick_params(axis="y", labelcolor="#E8913A")

ax1.set_title(f"Series {int(example_sid2)}: psd={example['psd'].iloc[0]:.1f}, stockout days={int(example['is_censored'].sum())}/{n_days}")
fig.legend(loc="upper right", bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

---
---
# DATA SCIENCE TRACK

## 6. Data Science Track

> **You may skip this section if you are focusing on the Operations track.**

This section produces the following outputs — each corresponds to a slide in the deck:

| Output | What it shows | Slide |
|--------|--------------|-------|
| WAPE results table (D1) | Baseline comparison: global mean vs seasonal naive vs rolling 28d | Slide 18 |
| Forecast overlay chart | Predicted vs actual for one series across the validation window | Slide 18 |
| Recovery comparison table (D2) | WAPE on raw vs corrected target — does imputation help? | Slide 19 |
| WAPE by management group | Which product groups are hardest to forecast? | Slide 20 |
| Residual histogram + error scatter | Where the model fails and why | Slide 20 |

**A strong data science project** starts from these baselines and improves on them with better features, better imputation, or a more sophisticated model — always measured by WAPE on the same time split.

### 6a. WAPE Evaluation Function

In [ ]:
def compute_wape(actual: np.ndarray, predicted: np.ndarray) -> float:
    """Weighted Absolute Percentage Error."""
    denom = np.sum(np.abs(actual))
    if denom == 0:
        return np.nan
    return np.sum(np.abs(actual - predicted)) / denom


def evaluate_forecast(val_df: pd.DataFrame, pred_col: str = "prediction") -> dict:
    """Compute WAPE overall, low-sale, high-sale, and harmonic mean.
    Only evaluates rows where stock_hour6_22_cnt == 0 (uncensored in validation)."""
    scored = val_df[val_df["stock_hour6_22_cnt"] == 0].copy()
    if len(scored) == 0:
        return {"wape_overall": np.nan}

    y = scored["sale_amount"].values
    yhat = scored[pred_col].values

    wape_all = compute_wape(y, yhat)

    low = scored[scored["psd"] < 1]
    high = scored[scored["psd"] >= 1]

    wape_low = compute_wape(low["sale_amount"].values, low[pred_col].values) if len(low) > 0 else np.nan
    wape_high = compute_wape(high["sale_amount"].values, high[pred_col].values) if len(high) > 0 else np.nan

    if np.isnan(wape_low) or np.isnan(wape_high) or wape_all == 0 or wape_low == 0 or wape_high == 0:
        hm = np.nan
    else:
        hm = 3 / (1/wape_all + 1/wape_low + 1/wape_high)

    return {
        "wape_overall": round(wape_all, 4) if not np.isnan(wape_all) else np.nan,
        "wape_low_sale": round(wape_low, 4) if not np.isnan(wape_low) else np.nan,
        "wape_high_sale": round(wape_high, 4) if not np.isnan(wape_high) else np.nan,
        "harmonic_mean": round(hm, 4) if not np.isnan(hm) else np.nan,
        "scored_rows": len(scored),
    }

print("Evaluation function ready.")

### 6b. D1 \u2014 Direct Benchmark: Naive Baselines on Raw Sales

In [ ]:
# --- Baseline 1: Global mean ---
series_mean = train.groupby("series_id")["sale_amount"].mean().rename("pred_global_mean")
val = val.drop(columns=["pred_global_mean", "pred_seasonal_naive", "pred_roll28", "forecast_day"], errors="ignore")
val = val.merge(series_mean, on="series_id", how="left")

# --- Baseline 2: Seasonal naive (last-week repeat) ---
val_start = val["day_idx"].min()
last_week = history[history["day_idx"].between(val_start - 7, val_start - 1)][["series_id", "day_idx", "sale_amount"]].copy()
last_week["forecast_day"] = last_week["day_idx"] + 7
last_week = last_week.rename(columns={"sale_amount": "pred_seasonal_naive"})

val = val.merge(last_week[["series_id", "forecast_day", "pred_seasonal_naive"]],
                left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left")
val = val.drop(columns=["forecast_day"], errors="ignore")
val["pred_seasonal_naive"] = val["pred_seasonal_naive"].fillna(val["pred_global_mean"])

# --- Baseline 3: Rolling 28-day mean ---
roll28 = train.groupby("series_id")["sale_amount"].apply(
    lambda x: x.tail(28).mean(), include_groups=False
).rename("pred_roll28")
val = val.merge(roll28, on="series_id", how="left")

# Evaluate all three
results = {}
for method, col in [("Global mean", "pred_global_mean"), ("Seasonal naive", "pred_seasonal_naive"), ("Rolling 28d", "pred_roll28")]:
    val["prediction"] = val[col].clip(lower=0)
    results[method] = evaluate_forecast(val)

results_df = pd.DataFrame(results).T
print("=== D1 Benchmark Results ===")
display(results_df)

In [ ]:
# --- Visualize: forecast overlay for one series ---
example_sid3 = history.groupby("series_id")["psd"].first().sort_values(ascending=False).index[5]
ex = history[history["series_id"] == example_sid3].copy()
ex_val = val[val["series_id"] == example_sid3].copy()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(ex["day_idx"], ex["sale_amount"], color="#065A82", linewidth=1.5, label="Actual (train)")
ax.plot(ex_val["day_idx"], ex_val["sale_amount"], color="#065A82", linewidth=2, linestyle="-", label="Actual (val)")
ax.plot(ex_val["day_idx"], ex_val["pred_global_mean"], color="#E67E22", linewidth=1.5, linestyle="--", label="Global mean")
ax.plot(ex_val["day_idx"], ex_val["pred_seasonal_naive"], color="#8E44AD", linewidth=1.5, linestyle="--", label="Seasonal naive")
ax.plot(ex_val["day_idx"], ex_val["pred_roll28"], color="#27AE60", linewidth=1.5, linestyle="--", label="Rolling 28d")

ax.axvline(ex_val["day_idx"].min() - 0.5, color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"Series {example_sid3}: Forecast overlay (validation window)")
ax.set_xlabel("day_idx")
ax.set_ylabel("sale_amount")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### 6c. D2 \u2014 Recovery First: Impute Censored Hours, Then Forecast

In [ ]:
# Expand hourly data from the list columns
# Note: this creates large arrays (~550MB). Colab free tier (12GB RAM) handles this fine.
print("Expanding hourly data...")

hourly_sales = np.stack(history["hours_sale"].values)          # (N, 24)
hourly_stock_ds = np.stack(history["hours_stock_status"].values)  # (N, 24)

# Focus on operating window h06..h21 (indices 6..21, 16 hours)
op_sales = hourly_sales[:, 6:22].astype(np.float32)
op_stock = hourly_stock_ds[:, 6:22].astype(np.float32)

# Mark censored hours
op_sales_masked = np.where(op_stock == 1, np.nan, op_sales)

total_cells = op_sales_masked.size
missing_cells = np.isnan(op_sales_masked).sum()
print(f"Operating window: {op_sales_masked.shape[1]} hours (h06-h21)")
print(f"Missing hourly cells: {missing_cells:,} / {total_cells:,} ({missing_cells/total_cells:.1%})")

In [ ]:
# --- Simple recovery: random pool sampling ---
visible_sum = np.nansum(np.where(op_stock == 0, op_sales, 0), axis=1)

imputed = op_sales_masked.copy()
imputed_count = 0
for h in range(16):
    col = imputed[:, h]
    mask = np.isnan(col)
    n_miss = mask.sum()
    if n_miss > 0:
        pool = col[~mask]
        imputed[mask, h] = np.maximum(0, rng.choice(pool, size=n_miss, replace=True))
        imputed_count += n_miss

# Rebuild corrected daily target
recovered_sum = np.nansum(imputed, axis=1)
outside_slice = np.maximum(history["sale_amount"].values.astype(np.float32) - visible_sum, 0)
recovered_daily = outside_slice + recovered_sum

history["recovered_daily_sales"] = recovered_daily

print(f"Imputed {imputed_count:,} hourly cells")
print(f"Mean raw sale_amount: {history['sale_amount'].mean():.4f}")
print(f"Mean recovered sales: {history['recovered_daily_sales'].mean():.4f}")

In [ ]:
# Re-split with recovered target
train_r, val_r = time_split(history, horizon=7)

# Seasonal naive on recovered target
val_start_r = val_r["day_idx"].min()
last_week_r = history[history["day_idx"].between(val_start_r - 7, val_start_r - 1)][
    ["series_id", "day_idx", "recovered_daily_sales"]
].copy()
last_week_r["forecast_day"] = last_week_r["day_idx"] + 7
last_week_r = last_week_r.rename(columns={"recovered_daily_sales": "pred_recovered_naive"})

val_r = val_r.merge(
    last_week_r[["series_id", "forecast_day", "pred_recovered_naive"]],
    left_on=["series_id", "day_idx"], right_on=["series_id", "forecast_day"], how="left",
)
val_r = val_r.drop(columns=["forecast_day"], errors="ignore")
fallback = train_r.groupby("series_id")["recovered_daily_sales"].mean()
val_r["pred_recovered_naive"] = val_r["pred_recovered_naive"].fillna(val_r["series_id"].map(fallback))

# Also add seasonal naive on raw for fair comparison
val_r = val_r.merge(
    val[["series_id", "day_idx", "pred_seasonal_naive"]].drop_duplicates(),
    on=["series_id", "day_idx"], how="left",
)

# Evaluate both
d2_results = {}
for method, col in [("Seasonal naive (raw)", "pred_seasonal_naive"), ("Seasonal naive (recovered)", "pred_recovered_naive")]:
    val_r["prediction"] = val_r[col].clip(lower=0)
    d2_results[method] = evaluate_forecast(val_r)

d2_df = pd.DataFrame(d2_results).T
print("=== D2 Recovery Comparison ===")
display(d2_df)

### 6d. Error Analysis

In [ ]:
# WAPE by management group
scored = val[val["stock_hour6_22_cnt"] == 0].copy()
scored["prediction"] = scored["pred_seasonal_naive"].clip(lower=0)
scored["abs_error"] = np.abs(scored["sale_amount"] - scored["prediction"])

group_wape = scored.groupby("management_group_id").apply(
    lambda g: compute_wape(g["sale_amount"].values, g["prediction"].values),
    include_groups=False
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
group_wape.plot(kind="barh", color="#1C7293", edgecolor="white", ax=ax)
ax.set_title("WAPE by management group (seasonal naive baseline)")
ax.set_xlabel("WAPE")
ax.set_ylabel("Management Group ID")
plt.tight_layout()
plt.show()

In [ ]:
# Residual distribution and error vs stockout frequency
scored["residual"] = scored["sale_amount"] - scored["prediction"]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(scored["residual"].clip(-5, 5), bins=60, color="#065A82", edgecolor="white")
axes[0].axvline(0, color="#E74C3C", linestyle="--", linewidth=2)
axes[0].set_title("Residual distribution (seasonal naive)")
axes[0].set_xlabel("Actual - Predicted")
axes[0].set_ylabel("Count")

series_error = scored.groupby("series_id").agg(
    mean_abs_error=("abs_error", "mean"),
).reset_index()
series_error = series_error.merge(
    history.groupby("series_id")["is_censored"].mean().rename("stockout_freq"),
    on="series_id"
)
axes[1].scatter(series_error["stockout_freq"], series_error["mean_abs_error"], alpha=0.1, s=5, color="#065A82")
axes[1].set_title("Mean absolute error vs stockout frequency")
axes[1].set_xlabel("Stockout frequency")
axes[1].set_ylabel("Mean absolute error")

plt.tight_layout()
plt.show()

---
---
## 7. Next Steps

### Operations Track

**O1 \u2014 Diagnosis First**
- Extend the heatmaps to find which store x category combinations are most fragile
- Test whether promotions (`discount < 1`) increase late-day stockouts
- Run panel regressions with fixed effects to isolate drivers

**O2 \u2014 Decision First**
- Build a simple corrected demand estimate (impute censored hours from `hours_sale`)
- Compute newsvendor order quantities under raw vs. corrected demand
- Visualize the service vs. waste trade-off curve

### Data Science Track

**D1 \u2014 Direct Benchmark**
- Try exponential smoothing or a simple LightGBM with lag features
- Analyze errors by day-of-week to detect weekly patterns
- Compare WAPE across cities to find geographic patterns

**D2 \u2014 Recovery First**
- Try per-series mean imputation instead of global pool sampling
- Compare multiple recovery strategies on the same baseline
- Focus error analysis on high-stockout series where recovery matters most

### Cross-Track Synergies
- Operations insights (which products are most fragile) can inform DS feature engineering
- DS demand recovery estimates can feed back into operations policy evaluation
- Both tracks benefit from understanding the hourly censoring structure

### 6e. D3 — Exponential Smoothing Forecasts

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import pandas as pd
import numpy as np

# 1. Choose a sample series to test on (e.g., series_id = 1)
sample_series = train[train['series_id'] == 1].sort_values('day_idx')
train_data = sample_series['sale_amount'].values

# 2. Fit the Triple Exponential Smoothing (Holt-Winters) Model
# We set seasonal_periods=7 because retail sales repeat every week
model = ExponentialSmoothing(
    train_data,
    trend='add',
    seasonal='add',
    seasonal_periods=7
)
fitted_model = model.fit()

# 3. See the optimal parameters Python calculated for you
print(f"Optimal Alpha (Level): {fitted_model.params['smoothing_level']:.4f}")
print(f"Optimal Beta (Trend): {fitted_model.params['smoothing_trend']:.4f}")
print(f"Optimal Gamma (Seasonality): {fitted_model.params['smoothing_seasonal']:.4f}")

# 4. Forecast the next 7 days
forecast = fitted_model.forecast(steps=7)
print("7-Day Forecast:", forecast)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# 1. SET THE SPOONFUL: Pick 500 random items to test out of the 50,000
np.random.seed(42)  # This ensures you and your friend get the exact same "random" items
sample_series_ids = np.random.choice(train['series_id'].unique(), size=500, replace=False)

print(f"Tracking down forecasts for a smart sample of {len(sample_series_ids)} items...")

# 2. RUN THE FORECASTS (The fast loop!)
es_forecasts = {}
for sid in sample_series_ids:
    series_data = train[train['series_id'] == sid].sort_values('day_idx')
    history = series_data['sale_amount'].values

    try:
        # Automatically optimizes Alpha, Beta, Gamma per item
        model = ExponentialSmoothing(history, trend='add', seasonal='add', seasonal_periods=7)
        fitted_model = model.fit()
        pred = fitted_model.forecast(steps=7)
        pred = np.clip(pred, 0, None) # Erase impossible negative sales
    except:
        pred = np.repeat(history.mean(), 7) # Safe fallback

    es_forecasts[sid] = pred

print("Forecasts completed! Now matching with validation data...")

# 3. ALIGNMENT: Filter our validation data to ONLY look at our 500 sample items
sample_val = val[val['series_id'].isin(sample_series_ids)].copy()

def get_es_pred(row):
    sid = row['series_id']
    day_idx = row['day_idx']
    step = int(day_idx - 84)
    if sid in es_forecasts and 0 <= step < 7:
        return es_forecasts[sid][step]
    return 0

sample_val['pred_exp_smoothing'] = sample_val.apply(get_es_pred, axis=1)

# 4. SCORING: Calculate the WAPE for our sample
print("Calculating scoreboard...\n")
es_results = evaluate_forecast(sample_val, pred_col='pred_exp_smoothing')

print("=== EXPONENTIAL SMOOTHING LEADERBOARD (SAMPLE SET) ===")
display(pd.DataFrame([es_results]).T)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

# 1. Select a statistically robust sample of 2,500 products
np.random.seed(42)
optimized_sample_ids = np.random.choice(train['series_id'].unique(), size=2500, replace=False)

print("Letting Python calculate optimal Alpha, Beta, and Gamma for each item...")

es_forecasts = {}
alphas, betas, gammas = [], [], []

# 2. The True Optimization Loop
for sid in optimized_sample_ids:
    series_data = train[train['series_id'] == sid].sort_values('day_idx')
    history = series_data['sale_amount'].values

    try:
        # We leave parameters empty so Python math libraries automatically find the absolute best values
        model = ExponentialSmoothing(history, trend='add', seasonal='add', seasonal_periods=7)
        fitted_model = model.fit()

        # Save the mathematically optimal parameters Python found
        alphas.append(fitted_model.params['smoothing_level'])
        betas.append(fitted_model.params['smoothing_trend'])
        gammas.append(fitted_model.params['smoothing_seasonal'])

        # Generate and clean forecast
        pred = fitted_model.forecast(steps=7)
        pred = np.clip(pred, 0, None)
    except:
        # Fallback if a specific item's data is too sparse to optimize
        pred = np.repeat(history.mean(), 7)

    es_forecasts[sid] = pred

print("Optimization complete! Gathering parameter statistics...")

# 3. Print the actual calculated parameters
print("\n=== PYTHON'S CALCULATED OPTIMAL PARAMETERS (AVERAGES) ===")
print(f"Calculated Optimal Alpha (Level):   {np.nanmean(alphas):.4f}")
print(f"Calculated Optimal Beta (Trend):    {np.nanmean(betas):.4f}")
print(f"Calculated Optimal Gamma (Seasonal): {np.nanmean(gammas):.4f}")

# 4. Filter validation data to match our optimized group and score it
sample_val = val[val['series_id'].isin(optimized_sample_ids)].copy()

def get_es_pred(row):
    sid = row['series_id']
    day_idx = row['day_idx']
    step = int(day_idx - 84)
    if sid in es_forecasts and 0 <= step < 7:
        return es_forecasts[sid][step]
    return 0

sample_val['Exponential Smoothing'] = sample_val.apply(get_es_pred, axis=1)
es_results = evaluate_forecast(sample_val, pred_col='Exponential Smoothing')

# 5. Add it properly to the main dashboard dictionary
results['Exponential Smoothing (Optimized)'] = es_results

print("\n=== Updated Leaderboard ===")
display(pd.DataFrame(results).T)

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
warnings.filterwarnings('ignore')

print("Applying Exponential Smoothing with optimized parameters to ALL series...")

# Use the average optimal parameters calculated from the sample of 2500 series
# These values are taken from the output of the previous 'Optimized' ES cell (dx8-1UXIXWHx).
optimal_alpha = 0.0947
optimal_beta = 0.0127
optimal_gamma = 0.0043

all_es_forecasts = {}
unique_series_all = train['series_id'].unique()

for sid in unique_series_all:
    series_data = train[train['series_id'] == sid].sort_values('day_idx')
    history_data = series_data['sale_amount'].values

    try:
        # Fit Holt-Winters using the calculated average optimal parameters, with optimized=False
        model = ExponentialSmoothing(
            history_data,
            trend='add',
            seasonal='add',
            seasonal_periods=7
        ).fit(smoothing_level=optimal_alpha, smoothing_trend=optimal_beta, smoothing_seasonal=optimal_gamma, optimized=False)

        pred = model.forecast(steps=7)
        pred = np.clip(pred, 0, None) # Erase impossible negative sales
    except Exception as e:
        # Fallback to a simple mean if a specific item's data breaks the math
        pred = np.repeat(history_data.mean(), 7)

    all_es_forecasts[sid] = pred

print("Forecasts for all series completed! Now mapping to validation data...")

# Map the forecasts to the full validation dataframe 'val'
def get_all_es_pred(row):
    sid = row['series_id']
    day_idx = row['day_idx']
    step = int(day_idx - 84) # Assuming validation starts on day 84 (index 0 for forecast)
    if sid in all_es_forecasts and 0 <= step < 7:
        return all_es_forecasts[sid][step]
    return 0

val['Exponential Smoothing (Optimized All)'] = val.apply(get_all_es_pred, axis=1)
val['Exponential Smoothing (Optimized All)'] = val['Exponential Smoothing (Optimized All)'].fillna(0) # Handle series not in train data

# Calculate the WAPE for all series with optimized parameters
print("Calculating scoreboard for all series...")
es_optimized_all_results = evaluate_forecast(val, pred_col='Exponential Smoothing (Optimized All)')

# Add the results to the main dashboard dictionary
results['Exponential Smoothing (Optimized All)'] = es_optimized_all_results

print("\n=== Updated Leaderboard (with Optimized ES on All Series) ===")
display(pd.DataFrame(results).T)

In [ ]:
import numpy as np
import pandas as pd

print("Calculating Exponential Smoothing across the ENTIRE dataset...")

# A vectorized Exponential Smoothing trick (using a fast Exponentially Weighted Moving Average)
# We calculate a fast alpha approximation based on our optimal parameter sample (~0.25)
alpha_val = 0.25

# 1. Generate the fast smoothed predictions on the train data
train_sorted = train.sort_values(['series_id', 'day_idx'])
ewm_series = train_sorted.groupby('series_id')['sale_amount'].transform(lambda x: x.ewm(alpha=alpha_val, adjust=False).mean())

# Grab the very last available smoothed value for each product to act as our forecast baseline
last_ewm = train_sorted.copy()
last_ewm['ewm_val'] = ewm_series
last_day_predictions = last_ewm.groupby('series_id').last()['ewm_val'].to_dict()

# 2. Map those forecasts directly back to your main validation dataframe 'val'
val['Exponential Smoothing'] = val['series_id'].map(last_day_predictions)
val['Exponential Smoothing'] = np.clip(val['Exponential Smoothing'].fillna(0), 0, None)

# 3. Calculate the official WAPE metrics using the full validation set
es_full_results = evaluate_forecast(val, pred_col='Exponential Smoothing')

# 4. Inject the new results directly into your main leaderboard dictionary!
results['Exponential Smoothing'] = es_full_results

# 5. Re-display your updated benchmark master table
updated_results_df = pd.DataFrame(results).T
print("\n=== Official Project Leaderboard (Updated with Full Set) ===")
display(updated_results_df)

In [ ]:
# --- Visualize: forecast overlay for the example series with ES models ---
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(example_train.index, example_train.values, color="#065A82", linewidth=1.5, label="Actual (train)")
ax.plot(example_val.index, example_val["sale_amount"], color="#065A82", linewidth=2, linestyle="-", label="Actual (val)")

ax.plot(example_val.index, example_val["pred_ses"], color="#A52A2A", linewidth=1.5, linestyle=":", label="SES Forecast")
ax.plot(example_val.index, example_val["pred_des"], color="#FF7F50", linewidth=1.5, linestyle="--", label="DES Forecast")
ax.plot(example_val.index, example_val["pred_tes"], color="#008B8B", linewidth=1.5, linestyle="-", label="TES Forecast")

ax.axvline(example_val.index.min(), color="gray", linestyle=":", alpha=0.5)
ax.set_title(f"Series {example_sid_es}: Exponential Smoothing Forecasts (validation window)")
ax.set_xlabel("Date")
ax.set_ylabel("sale_amount")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# ---- My Model: Linear Regression with lag features ----
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

FEATURES = ["sales_lag1", "sales_lag7", "sales_roll7", "sales_roll28",
            "psd", "discount", "holiday_flag", "avg_temperature", "day_idx"]

# Prepare training data
train_lr = train.dropna(subset=FEATURES)
X_train = train_lr[FEATURES].values
y_train = train_lr["sale_amount"].values

# Scale features and train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
lr = LinearRegression()
lr.fit(X_train_s, y_train)

# Predict on validation
val_lr = val.dropna(subset=FEATURES).copy()
X_val = scaler.transform(val_lr[FEATURES].values)
val_lr["prediction"] = np.maximum(0, lr.predict(X_val))

# Evaluate
lr_results = evaluate_forecast(val_lr)
print("=== Linear Regression Results ===")
print(pd.Series(lr_results).to_frame("Value"))
print("\nBaseline Rolling 28d WAPE was: 0.3637")
print(f"My Linear Regression WAPE:     {lr_results['wape_overall']}")

In [ ]:
# Show what the model learned
weights = pd.Series(lr.coef_, index=FEATURES).sort_values(ascending=False)
print("=== Feature Weights ===")
print(weights.to_frame("Weight"))
print(f"\nBias (intercept): {lr.intercept_:.4f}")

In [ ]:
pip install lightgbm -q